In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
create table if not exists brz_bank_transactions
using csv
options (
  header = "true",
  inferSchema = "true"
)
location 'abfss://project-dataset@finrisk3605359934826.dfs.core.windows.net/';

## Data Auditing

In [0]:
table_name = "brz_bank_transactions"
audit_df = spark.table(table_name)
row_count = audit_df.count()
column_count = len(audit_df.columns)

print(f"Auditing table: {table_name}")
print(f"Row count: {row_count:,}")
print(f"Column count: {column_count}")
display(audit_df.limit(5))


In [0]:
# Schema checks
# Verifying that the bronze table still matches the expected structure
# and highlights any missing, extra, or mistyped columns.

expected_schema = {
    "transaction_id": "string",
    "customer_id": "string",
    "transaction_date": "date",
    "transaction_time": "timestamp",
    "account_type": "string",
    "transaction_type": "string",
    "transaction_amount": "double",
    "transaction_direction": "string",
    "account_balance": "double",
    "merchant_category": "string",
    "state": "string",
    "credit_score": "int",
    "has_loan": "int",
    "loan_type": "string",
    "emi_amount": "double",
    "transaction_status": "string",
    "channel": "string",
    "kyc_status": "string",
    "is_fraud": "int",
    "transaction_hour": "int",
}

actual_schema = {field.name: field.dataType.simpleString() for field in audit_df.schema.fields}

schema_rows = [
    (
        column_name,
        expected_schema[column_name],
        actual_schema.get(column_name),
        expected_schema[column_name] == actual_schema.get(column_name),
    )
    for column_name in expected_schema
]

missing_columns = sorted(set(expected_schema) - set(actual_schema))
unexpected_columns = sorted(set(actual_schema) - set(expected_schema))

schema_check_df = spark.createDataFrame(
    schema_rows,
    ["column_name", "expected_type", "actual_type", "matches_expected_type"],
)

print(f"Missing expected columns: {missing_columns or 'None'}")
print(f"Unexpected columns: {unexpected_columns or 'None'}")
display(schema_check_df.orderBy(F.col("matches_expected_type").asc(), F.col("column_name").asc()))


In [0]:
# Missing value checks
# Strings are treated as missing when they are null or blank after trimming whitespace.

if row_count == 0:
    print("The table is empty, so missing value profiling cannot be computed.")
else:
    missing_exprs = []
    for column_name, data_type in audit_df.dtypes:
        if data_type == "string":
            condition = F.col(column_name).isNull() | (F.trim(F.col(column_name)) == "")
        else:
            condition = F.col(column_name).isNull()

        missing_exprs.append(
            F.sum(F.when(condition, 1).otherwise(0)).alias(column_name)
        )

    missing_summary = audit_df.agg(*missing_exprs)
    stack_expr = "stack({0}, {1}) as (column_name, missing_count)".format(
        len(audit_df.columns),
        ", ".join([f"'{column_name}', `{column_name}`" for column_name in audit_df.columns]),
    )

    missing_long = (
        missing_summary
        .selectExpr(stack_expr)
        .withColumn("missing_pct", F.round(F.col("missing_count") / F.lit(row_count) * 100, 2))
        .orderBy(F.desc("missing_count"), F.asc("column_name"))
    )

    display(missing_long)


In [0]:
# Duplicate checks
# Full-row duplicates indicate exact record duplication.
# Transaction ID duplicates indicate business-key duplication.

if row_count == 0:
    print("The table is empty, so duplicate checks cannot be computed.")
else:
    full_row_duplicate_count = row_count - audit_df.dropDuplicates().count()

    transaction_id_duplicates = (
        audit_df.groupBy("transaction_id")
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.desc("count"), F.asc("transaction_id"))
    )

    duplicate_transaction_ids = transaction_id_duplicates.count()

    print(f"Exact duplicate rows: {full_row_duplicate_count:,}")
    print(f"Duplicate transaction_id values: {duplicate_transaction_ids:,}")
    display(transaction_id_duplicates.limit(20))


In [0]:
# Invalid value checks
# These rules focus on clear business or data-quality violations and cross-field inconsistencies.

invalid_rules = [
    ("negative_transaction_amount", F.col("transaction_amount") < 0),
    ("negative_account_balance_potential_issue", F.col("account_balance") < 0),
    (
        "credit_score_outside_300_900",
        F.col("credit_score").isNotNull() & ((F.col("credit_score") < 300) | (F.col("credit_score") > 900)),
    ),
    (
        "transaction_hour_outside_0_23",
        F.col("transaction_hour").isNotNull() & ((F.col("transaction_hour") < 0) | (F.col("transaction_hour") > 23)),
    ),
    (
        "has_loan_not_binary",
        F.col("has_loan").isNotNull() & (~F.col("has_loan").isin(0, 1)),
    ),
    (
        "is_fraud_not_binary",
        F.col("is_fraud").isNotNull() & (~F.col("is_fraud").isin(0, 1)),
    ),
    (
        "unexpected_transaction_direction",
        F.col("transaction_direction").isNotNull() & (~F.col("transaction_direction").isin("Debit", "Credit")),
    ),
    (
        "loan_type_inconsistent_with_has_loan",
        ((F.col("has_loan") == 0) & F.col("loan_type").isNotNull() & (F.trim(F.lower(F.col("loan_type"))) != "none"))
        | ((F.col("has_loan") == 1) & (F.col("loan_type").isNull() | (F.trim(F.lower(F.col("loan_type"))) == "none"))),
    ),
    (
        "emi_amount_inconsistent_with_has_loan",
        ((F.col("has_loan") == 0) & (F.col("emi_amount") > 0))
        | ((F.col("has_loan") == 1) & (F.col("emi_amount").isNull() | (F.col("emi_amount") <= 0))),
    ),
]

if row_count == 0:
    print("The table is empty, so invalid value checks cannot be computed.")
else:
    invalid_summary = audit_df.agg(
        *[F.sum(F.when(condition, 1).otherwise(0)).alias(rule_name) for rule_name, condition in invalid_rules]
    )

    stack_expr = "stack({0}, {1}) as (rule_name, invalid_count)".format(
        len(invalid_rules),
        ", ".join([f"'{rule_name}', `{rule_name}`" for rule_name, _ in invalid_rules]),
    )

    invalid_long = (
        invalid_summary
        .selectExpr(stack_expr)
        .withColumn("invalid_pct", F.round(F.col("invalid_count") / F.lit(row_count) * 100, 2))
        .orderBy(F.desc("invalid_count"), F.asc("rule_name"))
    )

    display(invalid_long)


In [0]:
# Class imbalance checks
# The target column is is_fraud. This cell shows prevalence and imbalance ratio.

if row_count == 0:
    print("The table is empty, so class balance cannot be profiled.")
else:
    class_summary = (
        audit_df.groupBy("is_fraud")
        .count()
        .withColumn("class_pct", F.round(F.col("count") / F.lit(row_count) * 100, 4))
        .orderBy(F.col("is_fraud").asc())
    )

    class_counts = {row["is_fraud"]: row["count"] for row in class_summary.collect()}

    if len(class_counts) < 2:
        print("Only one class is present in is_fraud. This is a severe class imbalance issue for modeling.")
        imbalance_ratio = None
    else:
        minority_count = min(class_counts.values())
        majority_count = max(class_counts.values())
        imbalance_ratio = round(majority_count / minority_count, 2) if minority_count else None

    display(class_summary)
    print(f"Imbalance ratio (majority / minority): {imbalance_ratio}")


In [0]:
# Target leakage checks
# This is a heuristic audit. It looks for columns whose names imply post-outcome knowledge,
# explicitly checks suspected fields, and profiles how pure they are with respect to the target label.
# It also shows correlations across numerical features.

douted_fields = ["transaction_status", "kyc_status"]
candidate_leakage_columns = [column_name for column_name in douted_fields if column_name in audit_df.columns]

print(f"Leakage-prone columns from douted_fields: {candidate_leakage_columns or 'None'}")

leakage_rows = []
for column_name in candidate_leakage_columns:
    purity_row = (
        audit_df.groupBy(column_name)
        .agg(
            F.count("*").alias("rows"),
            F.avg(F.col("is_fraud").cast("double")).alias("fraud_rate"),
        )
        .filter(F.col("rows") >= 50)
        .agg(
            F.max(
                F.when(F.col("fraud_rate") >= 0.5, F.col("fraud_rate")).otherwise(F.lit(1.0) - F.col("fraud_rate"))
            ).alias("max_class_purity")
        )
        .collect()[0]
    )

    leakage_rows.append(
        (
            column_name,
            float(purity_row["max_class_purity"]) if purity_row["max_class_purity"] is not None else None,
        )
    )

    print(f"Top target split for {column_name}")
    display(
        audit_df.groupBy(column_name)
        .agg(
            F.count("*").alias("rows"),
            F.round(F.avg(F.col("is_fraud").cast("double")), 4).alias("fraud_rate"),
        )
        .orderBy(F.desc("rows"), F.asc(column_name))
        .limit(20)
    )

if leakage_rows:
    leakage_df = spark.createDataFrame(
        leakage_rows,
        ["column_name", "max_class_purity"],
    ).orderBy(F.desc("max_class_purity"))

    display(leakage_df)
else:
    print("No obvious leakage-prone column names were found. Review any post-event operational fields manually before modeling.")

numeric_columns = [
    column_name
    for column_name, data_type in audit_df.dtypes
    if data_type in {"int", "bigint", "float", "double", "decimal", "smallint", "tinyint"}
]

if len(numeric_columns) < 2:
    print("Not enough numerical features to compute correlations.")
else:
    correlation_rows = []
    for i, column_a in enumerate(numeric_columns):
        for column_b in numeric_columns[i + 1:]:
            corr_value = audit_df.stat.corr(column_a, column_b)
            correlation_rows.append((column_a, column_b, corr_value))

    correlation_df = spark.createDataFrame(
        correlation_rows,
        ["feature_1", "feature_2", "correlation"],
    ).withColumn("abs_correlation", F.abs(F.col("correlation")))

    print("Top numerical feature correlations")
    display(correlation_df.orderBy(F.desc("abs_correlation"), F.asc("feature_1"), F.asc("feature_2")))

    if "is_fraud" in numeric_columns:
        print("Correlation of numerical features with is_fraud")
        display(
            correlation_df
            .filter((F.col("feature_1") == "is_fraud") | (F.col("feature_2") == "is_fraud"))
            .orderBy(F.desc("abs_correlation"), F.asc("feature_1"), F.asc("feature_2"))
        )


In [0]:
# Privacy risk checks
# This cell flags direct identifiers, high-cardinality quasi-identifiers, and sensitive attributes.

privacy_keywords = [
    "name",
    "email",
    "phone",
    "address",
    "ssn",
    "aadhaar",
    "pan",
    "customer",
    "account",
    "transaction",
    "loan",
    "credit",
    "kyc",
    "state",
    "date",
    "time",
]

privacy_named_columns = [
    column_name
    for column_name in audit_df.columns
    if any(keyword in column_name.lower() for keyword in privacy_keywords)
]

if row_count == 0:
    print("The table is empty, so privacy profiling cannot be computed.")
else:
    privacy_summary = audit_df.agg(
        *[F.countDistinct(F.col(column_name)).alias(column_name) for column_name in audit_df.columns]
    )

    stack_expr = "stack({0}, {1}) as (column_name, distinct_count)".format(
        len(audit_df.columns),
        ", ".join([f"'{column_name}', `{column_name}`" for column_name in audit_df.columns]),
    )

    privacy_long = (
        privacy_summary
        .selectExpr(stack_expr)
        .withColumn("distinct_ratio", F.round(F.col("distinct_count") / F.lit(row_count), 4))
        .withColumn(
            "risk_reason",
            F.when(F.col("column_name").rlike("_id$"), F.lit("Likely identifier"))
            .when(F.col("column_name").isin(privacy_named_columns), F.lit("Sensitive or quasi-identifier by name"))
            .when(F.col("distinct_count") == row_count, F.lit("Unique per row"))
            .when(F.col("distinct_ratio") >= 0.90, F.lit("Very high cardinality"))
            .otherwise(F.lit("Lower immediate risk")),
        )
        .orderBy(F.desc("distinct_ratio"), F.asc("column_name"))
    )

    display(privacy_long.filter(F.col("risk_reason") != "Lower immediate risk"))


### Data Audition Report

* **Schema checks**: The schema is perfect. 
* **Missing values**: No missing value found. 
* **Duplicates**: No Duplicate Row, No duplicated Identifiers
* **Invalid values**: Even if there is `has_loan` the `loan_type` shows "None" in some records
* **Class imbalance**: `Legits` are 99.1% and `Frauds` are 0.89%. Highly imbalanced
* target leakage,
* privacy risks

In [0]:
audit_df.filter((F.col("has_loan") == 1) & (F.col("loan_type") == 'None')).show()

In [0]:
audit_df.groupBy("transaction_status").agg(F.count("*")).show()